In [1]:
import os
import json
import csv
import warnings
from collections import defaultdict
import pandas as pd
import requests
from sqlalchemy import create_engine, inspect, exc, text
import chromadb
from chromadb.utils import embedding_functions
from datasketch import MinHash, MinHashLSH
import time
import pickle
import numpy as np

# Suppress SQLAlchemy server version warnings
warnings.filterwarnings("ignore", category=exc.SAWarning)

In [25]:
class NL2SQLDataPipeline:
    def __init__(self, db_config=None, chroma_db_path='../data/schema_db', 
                 collection_name='schema_collection', excluded_csv_path=None,
                 lsh_index_path='../data/lsh_index.pkl'): # Added lsh_index_path
        """
        Initializes the NL2SQL Pipeline with database, ChromaDB, and LSH settings.
        :param excluded_csv_path: Path to a CSV file containing columns to be excluded.
        :param lsh_index_path: Path to serialize the LSH value index.
        """
        self.db_config = db_config or {
            "sql_server": os.getenv("SQL_SERVER", "localhost"),
            "sql_database": os.getenv("SQL_DATABASE", "IPMPBI"),
            "sql_user": os.getenv("SQL_USER", "sa"),
            "sql_password": os.getenv("SQL_PASSWORD", "YourStrongPassword123!"),
            "sql_port": int(os.getenv("SQL_PORT", "1433"))
        }
        self.chroma_db_path = chroma_db_path
        self.collection_name = collection_name
        self.lsh_index_path = lsh_index_path
        self.engine = None
        self.lsh_data = None  # Holds loaded LSH, num_perm, and config
        self._excluded_columns = self._load_excluded_columns(excluded_csv_path)

    def _get_engine(self):
        """Creates and returns the SQLAlchemy engine singleton."""
        if not self.engine:
            connection_uri = (
                f"mssql+pytds://{self.db_config['sql_user']}:{self.db_config['sql_password']}"
                f"@{self.db_config['sql_server']}:{self.db_config['sql_port']}/{self.db_config['sql_database']}"
            )
            self.engine = create_engine(connection_uri)
        return self.engine

    def get_schema_and_samples(self, target_tables=None, sample_rows=3):
        """
        Extracts the schema definitions and sample data, supporting custom schemas 
        like 'Report.vw_Contracts'.
        
        :param target_tables: List of table/view names. Can include schemas, e.g., ['Report.vw_Contracts']
        :param sample_rows: Number of sample rows to retrieve.
        """
        engine = self._get_engine()
        inspector = inspect(engine)
        
        if not target_tables:
            # If no tables are provided, default to all tables in the default schema (usually dbo)
            target_tables = inspector.get_table_names()
            
        extracted_data = {}
        
        for table_fullname in target_tables:
            # 1. Parse schema and table/view name
            if "." in table_fullname:
                schema, table_name = table_fullname.split(".", 1)
            else:
                schema, table_name = None, table_fullname
            
            # 2. Retrieve columns, PKs, and FKs using the schema parameter
            try:
                columns = inspector.get_columns(table_name, schema=schema)
                pk = inspector.get_pk_constraint(table_name, schema=schema)
                fks = inspector.get_foreign_keys(table_name, schema=schema)
            except Exception as e:
                # If inspection fails (e.g., table/view doesn't exist), log and skip
                extracted_data[table_fullname] = {
                    "ddl": f"-- Error inspecting {table_fullname}: {e}",
                    "samples": ""
                }
                continue

            ddl_lines = []
            for col in columns:
                col_name = col['name']
                col_type = col['type']
                nullable = "NULL" if col['nullable'] else "NOT NULL"
                default = f" DEFAULT {col['default']}" if col.get('default') is not None else ""
                ddl_lines.append(f"    {col_name} {col_type} {nullable}{default}")
            
            # Add Primary Keys (Views typically won't return PKs)
            if pk and pk.get('constrained_columns'):
                pk_cols = ", ".join(pk['constrained_columns'])
                ddl_lines.append(f"    PRIMARY KEY ({pk_cols})")
                
            # Add Foreign Keys (Views typically won't return FKs)
            for fk in fks:
                constrained = ", ".join(fk['constrained_columns'])
                referred_table = fk['referred_table']
                referred_cols = ", ".join(fk['referred_columns'])
                ddl_lines.append(f"    FOREIGN KEY ({constrained}) REFERENCES {referred_table}({referred_cols})")
                
            ddl_string = f"CREATE TABLE {table_fullname} (\n" + ",\n".join(ddl_lines) + "\n);"
            
            # 3. Fetch sample data using the fully qualified name
            # Properly escape schema and table/view name for SQL Server
            if schema:
                qualified_name = f"[{schema}].[{table_name}]"
            else:
                qualified_name = f"[{table_name}]"
                
            sample_markdown = ""
            try:
                query = f"SELECT TOP {sample_rows} * FROM {qualified_name}"
                df = pd.read_sql(query, engine)
                
                if df.empty:
                    sample_markdown = "-- No data found in this table/view."
                else:
                    sample_markdown = df.to_markdown(index=False)
            except Exception as e:
                sample_markdown = f"-- Error fetching sample data: {e}"
                
            extracted_data[table_fullname] = {
                "ddl": ddl_string,
                "samples": sample_markdown
            }
            
        return extracted_data

    def _load_excluded_columns(self, csv_file_path: str) -> set:
        """
        Loads table and column names from a CSV file into a set for quick exclusion checks.
        The set stores (table_name.lower(), column_name.lower()) for case-insensitive matching.
        """
        excluded_set = set()
        if not csv_file_path or not os.path.exists(csv_file_path):
            if csv_file_path:
                print(f"Warning: Exclusion CSV '{csv_file_path}' not found or path is empty. No columns will be excluded via CSV.")
            return excluded_set
        
        try:
            df = pd.read_csv(csv_file_path)
            df.columns = df.columns.str.replace('ï»¿', '').str.strip()
            
            table_col = next((c for c in df.columns if c.upper() in ['TABLE_NAME', 'TABLE', 'VIEW_NAME', 'VIEW']), None)
            column_col = next((c for c in df.columns if c.upper() in ['COLUMN_NAME', 'COLUMN', 'COL']), None)

            if not table_col or not column_col:
                print(f"Warning: Exclusion CSV '{csv_file_path}' must contain fields indicating table and column names. Columns found: {list(df.columns)}. No columns will be excluded via CSV.")
                return excluded_set

            for _, row in df.iterrows():
                table_name = str(row[table_col]).strip() if pd.notna(row[table_col]) else ""
                column_name = str(row[column_col]).strip() if pd.notna(row[column_col]) else ""
                if table_name and column_name:
                    excluded_set.add((table_name.lower(), column_name.lower()))
            print(f"Loaded {len(excluded_set)} column exclusions from '{csv_file_path}'.")
        except Exception as e:
            print(f"Error loading exclusion CSV '{csv_file_path}': {e}. No columns will be excluded via CSV.")
        return excluded_set

    # ------------------ New LSH Helper Methods ------------------

    def _get_shingles(self, text: str, k: int = 3) -> set:
        """
        Splits text into character k-shingles (substrings of length k) for spelling-robust matching.
        Example: "Acme" with k=3 -> {'acm', 'cme'}
        """
        if not text:
            return set()
        text = text.lower().strip()
        if len(text) < k:
            return {text}
        return {text[i:i+k] for i in range(len(text) - k + 1)}

    def _jaccard_similarity(self, set_a: set, set_b: set) -> float:
        """Computes basic Jaccard similarity between two sets."""
        union_len = len(set_a.union(set_b))
        if union_len == 0:
            return 0.0
        return len(set_a.intersection(set_b)) / union_len

    def build_value_lsh_index(self, lsh_targets: dict, threshold: float = 0.4, 
                              num_perm: int = 128, shingle_size: int = 3, sample_limit: int = 10000):
        """
        Extracts unique values from specified columns in the database and indexes them using MinHash LSH.
        Saves the resulting index to self.lsh_index_path.
        
        :param lsh_targets: Dict structure: { "schema_name": { "table_or_view": ["column1", "column2"] } }
        :param threshold: The Jaccard similarity threshold for the LSH index.
        :param num_perm: Number of permutations for the MinHash.
        :param shingle_size: The character length of the shingles (k-mer value).
        :param sample_limit: Limit on unique values indexed per column to control file sizes and memory.
        """

        engine = self._get_engine()
        lsh = MinHashLSH(threshold=threshold, num_perm=num_perm)
        total_indexed = 0

        try:
            start = time.perf_counter()
            with engine.connect() as conn:
                for schema, tables in lsh_targets.items():
                    for table, columns in tables.items():
                        for col in columns:
                            # Exclusion check matching existing pipeline logic
                            if (table.lower(), col.lower()) in self._excluded_columns:
                                print(f"   - Skipping excluded LSH target column: {schema}.{table}.{col}")
                                continue

                            print(f"Fetching and indexing values for: {schema}.{table}.{col}...")
                            try:
                                query = text(f'SELECT DISTINCT TOP {sample_limit} "{col}" FROM "{schema}"."{table}" WHERE "{col}" IS NOT NULL')
                                result = conn.execute(query).fetchall()
                                values = [str(row[0]).strip() for row in result if row[0] is not None]

                                for val in values:
                                    if not val:
                                        continue
                                    
                                    m = MinHash(num_perm=num_perm)
                                    shingles = self._get_shingles(val, k=shingle_size)
                                    for s in shingles:
                                        m.update(s.encode('utf-8'))
                                    
                                    # Create a serialized identifier to store source column and value
                                    key = f"{schema}|{table}|{col}|{val}"
                                    lsh.insert(key, m)
                                    total_indexed += 1
                                    
                            except Exception as col_err:
                                print(f"   ! Values skip for {schema}.{table}.{col}: {col_err}")
            end = time.perf_counter()

            # Store the configuration and LSH index as a single serializable dictionary
            self.lsh_data = {
                "lsh": lsh,
                "num_perm": num_perm,
                "shingle_size": shingle_size,
                "threshold": threshold
            }
            
            output_dir = os.path.dirname(self.lsh_index_path)
            if output_dir:
                os.makedirs(output_dir, exist_ok=True)
                
            with open(self.lsh_index_path, 'wb') as f:
                pickle.dump(self.lsh_data, f)
                
            print(f"\nLSH indexing complete! Processed {total_indexed} total elements in {end-start} seconds. Index saved to '{self.lsh_index_path}'.")
            
        except Exception as e:
            print(f"Error during LSH indexing: {e}")
        finally:
            if getattr(self, 'engine', None):
                self.engine.dispose()
                self.engine = None

    def load_lsh_index(self) -> bool:
        """
        Loads the LSH index from self.lsh_index_path.
        Returns True if successful, False otherwise.
        """
        if not self.lsh_index_path or not os.path.exists(self.lsh_index_path):
            print(f"No LSH index found at '{self.lsh_index_path}'. Please run build_value_lsh_index first.")
            return False
        
        try:
            with open(self.lsh_index_path, 'rb') as f:
                self.lsh_data = pickle.load(f)
            print(f"Successfully loaded LSH index from '{self.lsh_index_path}'.")
            return True
        except Exception as e:
            print(f"Error loading LSH index from '{self.lsh_index_path}': {e}")
            return False

    def query_lsh(self, query_text: str) -> list:
        """
        Queries the loaded LSH index with a term to find similar values in database columns.
        Also calculates the exact Jaccard similarity between query shingles and candidate shingles
        to return a cleanly ranked list.
        
        :param query_text: String value to look up (e.g., misspelled word, abbreviation)
        :return: A list of dict matches sorted by Jaccard similarity in descending order.
        """
        if not self.lsh_data:
            if not self.load_lsh_index():
                return []
        
        try:
            from datasketch import MinHash
        except ImportError:
            print("Error: 'datasketch' is required to query. Run 'pip install datasketch'.")
            return []

        lsh = self.lsh_data["lsh"]
        num_perm = self.lsh_data["num_perm"]
        shingle_size = self.lsh_data["shingle_size"]

        # Prepare query shingles and MinHash
        query_shingles = self._get_shingles(query_text, k=shingle_size)
        if not query_shingles:
            return []
            
        m = MinHash(num_perm=num_perm)
        for s in query_shingles:
            m.update(s.encode('utf-8'))

        # Query candidate list
        raw_results = lsh.query(m)
        
        parsed_results = []
        for res in raw_results:
            try:
                schema, table, col, val = res.split('|', 3)
                val_shingles = self._get_shingles(val, k=shingle_size)
                similarity = self._jaccard_similarity(query_shingles, val_shingles)
                
                parsed_results.append({
                    "schema": schema,
                    "table": table,
                    "column": col,
                    "value": val,
                    "similarity": round(similarity, 4)
                })
            except ValueError:
                # Fallback if key structure differs
                parsed_results.append({
                    "raw_key": res,
                    "similarity": 0.0
                })

        # Sort exact candidate matches descending by Jaccard similarity
        parsed_results.sort(key=lambda x: x["similarity"], reverse=True)
        return parsed_results

    # ------------------ End LSH Methods ------------------
    def export_joinable_views_csv(self, csv_output_path: str = '../data/joinable_views.csv', schema_name: str = 'Report'):
        """
        Executes a metadata query to find columns in different views within the same schema
        that share similar names and key/ID suffixes, then exports the results to a CSV file.
        
        :param csv_output_path: Target path for the generated CSV file.
        :param schema_name: The database schema to filter views by (defaults to 'Report').
        """
        engine = self._get_engine()
        
        # Use a parameterized query. Note that 'USE' and 'GO' are omitted as the connection 
        # is already bound to the active database.
        query = text("""
            SELECT 
                c1.name AS ColumnInViewA,
                c2.name AS ColumnInViewB,
                t1.name AS DataType,
                s1.name AS SchemaNameA,
                v1.name AS ViewNameA,
                s2.name AS SchemaNameB,
                v2.name AS ViewNameB,
                'SELECT * FROM ' + QUOTENAME(s1.name) + '.' + QUOTENAME(v1.name) + ' AS a INNER JOIN ' + QUOTENAME(s2.name) + '.' + QUOTENAME(v2.name) + ' AS b ON a.' + QUOTENAME(c1.name) + ' = b.' + QUOTENAME(c2.name) AS SampleJoinQuery
            FROM 
                sys.columns c1
            INNER JOIN 
                sys.views v1 ON c1.object_id = v1.object_id
            INNER JOIN
                sys.schemas s1 ON v1.schema_id = s1.schema_id
            INNER JOIN 
                sys.columns c2 ON 
                    LOWER(REPLACE(c1.name, '_', '')) = LOWER(REPLACE(c2.name, '_', '')) 
                    AND c1.object_id <> c2.object_id 
            INNER JOIN 
                sys.views v2 ON c2.object_id = v2.object_id
            INNER JOIN
                sys.schemas s2 ON v2.schema_id = s2.schema_id
            INNER JOIN 
                sys.types t1 ON c1.system_type_id = t1.system_type_id AND c1.user_type_id = t1.user_type_id
            WHERE 
                -- Limit both views strictly to the specified schema
                s1.name = :schema_name
                AND s2.name = :schema_name
                -- Prevent matching views to themselves and hide duplicate reverse pairs
                AND v1.name < v2.name 
                -- Filter specifically for common key/ID naming patterns
                AND (
                    c1.name LIKE '%ID' 
                    OR c1.name LIKE '%Key' 
                    OR c1.name LIKE '%PK' 
                    OR c1.name LIKE '%FK' 
                    OR c1.name LIKE '%Code'
                )
            ORDER BY 
                ColumnInViewA, ViewNameA;
        """)

        try:
            # Ensure target directory exists
            output_dir = os.path.dirname(csv_output_path)
            if output_dir:
                os.makedirs(output_dir, exist_ok=True)

            print(f"Querying potential join relationships for schema '{schema_name}'...")
            
            with engine.connect() as conn:
                df = pd.read_sql(query, conn, params={"schema_name": schema_name})
            
            if df.empty:
                print(f"No joinable columns/views matching criteria found in schema '{schema_name}'.")
            
            df.to_csv(csv_output_path, index=False, encoding='utf-8')
            print(f"Saved joinable views report to '{csv_output_path}' ({len(df)} records written).")
            return df

        except Exception as e:
            print(f"Error exporting joinable views CSV: {e}")
            return None
        finally:
            if getattr(self, 'engine', None):
                self.engine.dispose()
                self.engine = None
    # ------------------ View Embedding Generation Methods ------------------

    def build_view_embeddings_from_dict(self, view_desc_dict: dict, 
                                        npy_path: str = '../data/view_embeddings.npy', 
                                        names_path: str = '../data/view_names.pkl'):
        """
        Generates vector embeddings for a mapping of view names to descriptions 
        and saves them to disk.
        """
        if not view_desc_dict:
            print("Error: The provided view descriptions dictionary is empty.")
            return

        view_names = []
        texts_to_embed = []

        for view_name, description in view_desc_dict.items():
            view_names.append(view_name)
            # Use description if populated, otherwise use a placeholder string as a fallback
            embed_text = description.strip() if description else f"Database view or table named {view_name}."
            texts_to_embed.append(embed_text)

        print(f"Generating embeddings for {len(view_names)} views...")
        embedding_fn = self.get_embedding_function()

        try:
            # Generate the vector representations using the configured model
            raw_embeddings = embedding_fn(texts_to_embed)
            
            # Convert list of embeddings to a structured numpy array
            view_embeddings = np.array(raw_embeddings, dtype=np.float32)

            # Ensure parent directories exist
            for path in [npy_path, names_path]:
                output_dir = os.path.dirname(path)
                if output_dir:
                    os.makedirs(output_dir, exist_ok=True)

            # Save the .npy file containing vectors
            np.save(npy_path, view_embeddings)
            print(f"Saved view embeddings to '{npy_path}' (Shape: {view_embeddings.shape}).")

            # Save the corresponding .pkl file containing view names
            with open(names_path, 'wb') as f:
                pickle.dump(view_names, f)
            print(f"Saved view names to '{names_path}'.")

        except Exception as e:
            print(f"Error generating or saving view embeddings: {e}")

    def build_view_embeddings_from_csv(self, csv_file_path: str, 
                                        npy_path: str = '../data/view_embeddings.npy', 
                                        names_path: str = '../data/view_names.pkl'):
        """
        Reads view names and descriptions from a CSV file, generates vector embeddings,
        and saves the outputs to .npy and .pkl formats.
        """
        try:
            df = pd.read_csv(csv_file_path)
            df.columns = df.columns.str.replace('ï»¿', '').str.strip()
        except FileNotFoundError:
            print(f"Error: The file '{csv_file_path}' was not found.")
            return
        except Exception as e:
            print(f"Error loading CSV file: {e}")
            return

        view_col = next((c for c in df.columns if c.upper() in ['VIEW_NAME', 'VIEW NAME', 'VIEW', 'TABLE_NAME', 'TABLE']), None)
        desc_col = next((c for c in df.columns if c.upper() in ['DESCRIPTION', 'DESC']), None)

        if not view_col or not desc_col:
            print(f"Error: CSV must contain fields indicating the view name and description. Columns found: {list(df.columns)}")
            return

        view_desc_map = {}
        for _, row in df.iterrows():
            if pd.isna(row[view_col]):
                continue
            
            view_name = str(row[view_col]).strip()
            description = str(row[desc_col]).strip() if pd.notna(row[desc_col]) else ""
            view_desc_map[view_name] = description

        self.build_view_embeddings_from_dict(view_desc_map, npy_path, names_path)

    # ------------------ End Embedding Methods ------------------

    def convert_view_desc_csv_to_json(self, csv_file_path: str, json_output_path: str) -> dict:
        """
        Converts a CSV with view/table descriptions into a mapped dictionary and saves it as a JSON file.
        """
        try:
            df = pd.read_csv(csv_file_path)
            df.columns = df.columns.str.replace('ï»¿', '').str.strip()
        except FileNotFoundError:
            print(f"Error: The file '{csv_file_path}' was not found.")
            return {}
        except Exception as e:
            print(f"Error loading CSV file: {e}")
            return {}

        view_col = next((c for c in df.columns if c.upper() in ['VIEW_NAME', 'VIEW NAME', 'VIEW', 'TABLE_NAME', 'TABLE']), None)
        desc_col = next((c for c in df.columns if c.upper() in ['DESCRIPTION', 'DESC']), None)

        if not view_col or not desc_col:
            print(f"Error: CSV must contain fields indicating the view name and its description. Columns found: {list(df.columns)}")
            return {}

        view_desc_map = {}
        for _, row in df.iterrows():
            if pd.isna(row[view_col]):
                continue
            
            view_name = str(row[view_col]).strip()
            description = str(row[desc_col]).strip() if pd.notna(row[desc_col]) else ""
            view_desc_map[view_name] = description

        try:
            output_dir = os.path.dirname(json_output_path)
            if output_dir:
                os.makedirs(output_dir, exist_ok=True)
                
            with open(json_output_path, 'w', encoding='utf-8') as f:
                json.dump(view_desc_map, f, indent=4, ensure_ascii=False)
            print(f"Successfully converted CSV and saved JSON to: '{json_output_path}'")
        except Exception as e:
            print(f"Error saving JSON to path '{json_output_path}': {e}")

        return view_desc_map

    def convert_column_desc_csv_to_jsonl(self, csv_file_path: str, jsonl_output_path: str) -> dict:
        """
        Converts a column description CSV into a structured JSONL (JSON Lines) file.
        Groups columns under their respective table/view names and skips records with empty column names
        or those present in the exclusion list.
        """
        try:
            df = pd.read_csv(csv_file_path)
            df.columns = df.columns.str.replace('ï»¿', '').str.strip()
        except FileNotFoundError:
            print(f"Error: The file '{csv_file_path}' was not found.")
            return {}
        except Exception as e:
            print(f"Error loading CSV file: {e}")
            return {}

        table_col = next((c for c in df.columns if c.upper() in ['TABLE_NAME', 'TABLE', 'VIEW_NAME', 'VIEW']), None)
        column_col = next((c for c in df.columns if c.upper() in ['COLUMN_NAME', 'COLUMN', 'COL']), None)
        desc_col = next((c for c in df.columns if c.upper() in ['DESCRIPTION', 'DESC']), None)

        if not table_col or not column_col or not desc_col:
            print(f"Error: CSV must contain fields indicating the Table Name, Column Name, and Description. Columns found: {list(df.columns)}")
            return {}

        aggregated_data = defaultdict(dict)
        for _, row in df.iterrows():
            if pd.isna(row[table_col]) or pd.isna(row[column_col]):
                continue
            
            table_name = str(row[table_col]).strip()
            column_name = str(row[column_col]).strip()

            if (table_name.lower(), column_name.lower()) in self._excluded_columns:
                print(f"   - Skipping excluded column from CSV conversion: {table_name}.{column_name}")
                continue

            description = str(row[desc_col]).strip() if pd.notna(row[desc_col]) else ""

            if not table_name or not column_name:
                continue

            aggregated_data[table_name][column_name] = description

        try:
            output_dir = os.path.dirname(jsonl_output_path)
            if output_dir:
                os.makedirs(output_dir, exist_ok=True)
                
            with open(jsonl_output_path, 'w', encoding='utf-8') as f:
                for table_name, columns in aggregated_data.items():
                    line_data = {
                        "view_name": table_name,
                        "columns": columns
                    }
                    f.write(json.dumps(line_data, ensure_ascii=False) + '\n')
            print(f"Successfully converted CSV and saved JSONL to: '{jsonl_output_path}'")
        except Exception as e:
            print(f"Error saving JSONL to path '{jsonl_output_path}': {e}")

        return dict(aggregated_data)

    def extract_database_metadata(self, schema_names=['Report'], output_dir='./metadata_output', columns_desc_csv_path='./test'):
            """
            Connects to the SQL Server, extracts schema data for multiple schemas, and identifies 
            categorical values (<=10) or examples (>10). Saves metadata as JSON.
            Excludes columns specified in the exclusion list.
            Also creates a './view_descriptions.csv' template containing the list of available tables.
            """
            if isinstance(schema_names, str):
                schema_names = [schema_names]
    
            engine = self._get_engine()
            metadata_registry = {}
            all_tables = []
    
            try:
                inspector = inspect(engine)
                os.makedirs(output_dir, exist_ok=True)
    
                with engine.connect() as conn:
                    for schema_name in schema_names:
                        print(f"\n--- Processing schema '{schema_name}' ---")
                        
                        views = inspector.get_view_names(schema=schema_name)
                        
                        if not views:
                            print(f"No views found in schema '{schema_name}'. Trying tables...")
                            views = inspector.get_table_names(schema=schema_name)
                            if not views:
                                print(f"No tables or views found in schema '{schema_name}'. Skipping.")
                                continue
    
                        print(f"Found {len(views)} objects in schema '{schema_name}'. Extracting metadata and value samples...")
    
                        for view_name in views:
                            columns_raw = inspector.get_columns(view_name, schema=schema_name)
                            view_data = []
    
                            for col in columns_raw:
                                col_name = col['name']
    
                                if (view_name.lower(), col_name.lower()) in self._excluded_columns:
                                    print(f"   - Skipping excluded column from database metadata: {schema_name}.{view_name}.{col_name}")
                                    continue
    
                                val_info = {}
                                
                                try:
                                    query = text(f'SELECT DISTINCT TOP 100 "{col_name}" FROM "{schema_name}"."{view_name}"')
                                    result = conn.execute(query).fetchall()
                                    raw_values = [row[0] for row in result if row[0] is not None]
                                    
                                    if len(raw_values) <= 10:
                                        val_info["unique_values"] = [
                                            str(v) if not isinstance(v, (int, float, bool)) else v for v in raw_values
                                        ]
                                    else:
                                        val_info["example_values"] = [
                                            str(v) if not isinstance(v, (int, float, bool)) else v for v in raw_values[:2]
                                        ]
                                except Exception as col_err:
                                    val_info = {"error": "Could not fetch values"}
                                    print(f"   ! Values skip for {schema_name}.{view_name}.{col_name}: {col_err}")
    
                                column_entry = {
                                    "name": col_name,
                                    "type": str(col['type']),
                                    "nullable": col['nullable'],
                                    "default": str(col['default']) if col.get('default') else None,
                                }
                                column_entry.update(val_info)
                                view_data.append(column_entry)
    
                            if view_data:
                                table_identifier = f"{schema_name}.{view_name}"
                                metadata_registry[table_identifier] = view_data
                                all_tables.append(table_identifier)
                                
                                file_path = os.path.join(output_dir, f"{schema_name}_{view_name}.json")
                                with open(file_path, 'w', encoding='utf-8') as f:
                                    json.dump(view_data, f, indent=4, ensure_ascii=False)
                                
                                print(f" - Saved: {file_path}")
                            else:
                                print(f" - Skipped saving empty metadata for view '{view_name}' (all columns excluded).")
    
                # Save the list of identified tables/views to './view_descriptions.csv'
                if all_tables:
                    try:
                        df_views = pd.DataFrame({
                            "VIEW_NAME": all_tables,
                            "DESCRIPTION": [""] * len(all_tables)
                        })
                        df_views.to_csv(os.path.join(columns_desc_csv_path, "view_descriptions.csv"), index=False, encoding='utf-8')
                        print(f"\nSaved list of available tables to './view_descriptions.csv'")
                    except Exception as csv_err:
                        print(f"\nError saving view descriptions CSV: {csv_err}")
    
                print(f"\nDatabase extraction complete! Files are saved in '{output_dir}'.")
                return metadata_registry
    
            except Exception as e:
                print(f"Database error: {e}")
                return metadata_registry 
            finally:
                if getattr(self, 'engine', None):
                    self.engine.dispose()
                    self.engine = None
                
    def export_column_desc_template(self, metadata_registry: dict, csv_output_path: str):
        """
        Creates a CSV template with TABLE_NAME, Column_Name, and an empty Description column
        based on the extracted database metadata, excluding columns in the exclusion list.
        """
        if not metadata_registry:
            print("No metadata registry available to export to CSV.")
            return

        rows = []
        for table_name, columns in metadata_registry.items():
            for col in columns:
                col_name = col["name"]
                if (table_name.lower(), col_name.lower()) in self._excluded_columns:
                    continue
                
                rows.append({
                    "TABLE_NAME": table_name,
                    "Column_Name": col_name,
                    "Description": ""
                })

        try:
            output_dir = os.path.dirname(csv_output_path)
            if output_dir:
                os.makedirs(output_dir, exist_ok=True)

            df = pd.DataFrame(rows)
            df.to_csv(csv_output_path, index=False, encoding='utf-8')
            print(f"Successfully generated column description template: '{csv_output_path}'")
        except Exception as e:
            print(f"Error generating CSV template file: {e}")

    def get_embedding_function(self, ollama_model="embeddinggemma", hf_model="google/embeddinggemma-300m", timeout=None):
        """Checks for Ollama running locally; falls back to Hugging Face."""
        ollama_url = "http://localhost:11434"
        try:
            response = requests.get(ollama_url, timeout=2)
            if response.status_code == 200:
                print(f"Ollama detected. Using local model: {ollama_model}")
                if timeout:
                    return embedding_functions.OllamaEmbeddingFunction(
                        url=f"{ollama_url}/api/embeddings",
                        model_name=ollama_model, 
                        timeout=timeout
                    )
                else:
                    return embedding_functions.OllamaEmbeddingFunction(
                        url=f"{ollama_url}/api/embeddings",
                        model_name=ollama_model
                    )
        except requests.exceptions.ConnectionError:
            print("Ollama not found. Falling back to Hugging Face API.")

        return embedding_functions.HuggingFaceEmbeddingFunction(
            api_key=os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACEHUB_API_TOKEN"),
            model_name=hf_model
        )

    def create_chroma_db_from_csv(self, csv_file_path: str, batch_size: int = 100):
        """
        Creates/populates ChromaDB from a schema CSV.
        """
        try:
            df = pd.read_csv(csv_file_path)
            df.columns = df.columns.str.replace('ï»¿', '').str.strip()
        except FileNotFoundError:
            print(f"Error: The file '{csv_file_path}' was not found.")
            return

        table_col = table_col = next((c for c in df.columns if c.upper() in ['TABLE_NAME', 'TABLE']), None)
        column_col = next((c for c in df.columns if c.upper() in ['COLUMN_NAME', 'COLUMN']), None)
        desc_col = next((c for c in df.columns if c.upper() in ['DESCRIPTION', 'DESC']), None)

        if not table_col or not column_col:
            print(f"Error: CSV must contain table and column name fields. Columns found: {list(df.columns)}")
            return

        print(f"Initializing ChromaDB client at '{self.chroma_db_path}'...")
        client = chromadb.PersistentClient(path=self.chroma_db_path)
        embedding_model = self.get_embedding_function()
        collection = client.get_or_create_collection(name=self.collection_name, embedding_function=embedding_model)

        documents, metadatas, ids = [], [], []

        print("Processing CSV records...")
        for index, row in df.iterrows():
            view_name = str(row[table_col])
            column_name = str(row[column_col])

            if (view_name.lower(), column_name.lower()) in self._excluded_columns:
                print(f"   - Skipping excluded column for ChromaDB from CSV: {view_name}.{column_name}")
                continue

            if desc_col and pd.notna(row[desc_col]):
                doc_text = str(row[desc_col])
            else:
                doc_text = f"Key / identifier column '{column_name}' in database table/view '{view_name}'."

            documents.append(doc_text)
            metadatas.append({
                'view_name': view_name,
                'view_column': column_name,
                'source': 'csv_import'
            })
            ids.append(f"csv_{view_name}_{column_name}_{index}")

        self._insert_batches(collection, documents, metadatas, ids, batch_size)

    def create_chroma_db_from_metadata(self, metadata_registry: dict, view_desc_dict: dict = None, batch_size: int = 100):
        """
        Populates ChromaDB directly from live database schema metadata.
        """
        if not metadata_registry:
            print("No database metadata to load into ChromaDB.")
            return

        print(f"Initializing ChromaDB client at '{self.chroma_db_path}'...")
        client = chromadb.PersistentClient(path=self.chroma_db_path)
        embedding_model = self.get_embedding_function(timeout=300)
        collection = client.get_or_create_collection(name=self.collection_name, embedding_function=embedding_model)

        documents, metadatas, ids = [], [], []

        for view_name, columns in metadata_registry.items():
            view_desc = view_desc_dict.get(view_name, "") if view_desc_dict else ""
            view_context_str = f" Context/Description: {view_desc}" if view_desc else ""

            for idx, col in enumerate(columns):
                col_name = col['name']

                if (view_name.lower(), col_name.lower()) in self._excluded_columns:
                    print(f"   - Skipping excluded column for ChromaDB from metadata: {view_name}.{col_name}")
                    continue

                col_type = col['type']
                nullable = "nullable" if col['nullable'] else "non-nullable"
                
                sample_vals = col.get('unique_values') or col.get('example_values') or []
                sample_str = ", ".join([f"'{v}'" for v in sample_vals]) if sample_vals else "none"

                doc_text = (
                    f"Column '{col_name}' in table '{view_name}'.{view_context_str} "
                    f"Data Type: {col_type} ({nullable}). "
                    f"Sample/Categorical Values: [{sample_str}]."
                )

                documents.append(doc_text)
                metadatas.append({
                    'view_name': view_name,
                    'view_column': col_name,
                    'source': 'live_db_metadata'
                })
                ids.append(f"db_{view_name}_{col_name}_{idx}")

        self._insert_batches(collection, documents, metadatas, ids, batch_size)

    def _insert_batches(self, collection, documents, metadatas, ids, batch_size):
        """Internal helper to insert records into ChromaDB in chunks."""
        if not documents:
            print("No documents to insert into ChromaDB.")
            return

        print(f"Writing {len(documents)} items in batches of {batch_size}...")
        for i in range(0, len(documents), batch_size):
            print(f"Sample {i} of {len(documents)}")
            collection.add(
                documents=documents[i : i + batch_size],
                metadatas=metadatas[i : i + batch_size],
                ids=ids[i : i + batch_size]
            )
        print(f"Success! Total documents in vector database: {collection.count()}")

In [26]:
EXCLUDED_CSV = '../docs/to_be_exluded.csv' 

pipeline = NL2SQLDataPipeline(
    chroma_db_path='../data/clean/schema_db',
    collection_name='schema_collection',
    lsh_index_path='../data/clean/lsh_index.pkl',
    excluded_csv_path=EXCLUDED_CSV # Pass the exclusion file path here
)

Loaded 199 column exclusions from '../docs/to_be_exluded.csv'.


In [21]:
# 3. Define the destination path for the CSV output
output_path = '../data/clean/IDColumns.csv'

# 4. Invoke the method
# This will query the metadata and save the CSV to the specified path
results_df = pipeline.export_joinable_views_csv(
    csv_output_path=output_path,
    schema_name='Report'
)

Querying potential join relationships for schema 'Report'...
Saved joinable views report to '../data/clean/IDColumns.csv' (536 records written).


In [17]:
tables_to_generate = [
    "Report.vw_CBS",
    "Report.vw_Contracts",
    "Report.vw_DataFormItem",
    "Report.vw_Invoice",
    "Report.vw_Issues",
    "Report.vw_Payments",
    "Report.vw_Project",
    "Report.vw_ProjectPlanProgress",
    "Report.vw_Projects",
    "Report.vw_ProjectScurve",
    "Report.vw_ResourceUsage",
    "Report.vw_Risks"
]

# Extract metadata and samples
db_info = pipeline.get_schema_and_samples(target_tables=tables_to_generate, sample_rows=3)

# Build the prompt
prompt_parts = [
    "You are a database assistant. Generate realistic SQL INSERT statements for SQL Server.",
    "Please generate 10 new, realistic records for each of the tables described below.",
    "Ensure correct referential integrity between tables.\n"
]

for table_name, data in db_info.items():
    prompt_parts.append(f"### Table: {table_name}")
    prompt_parts.append("#### Schema:")
    prompt_parts.append(f"```sql\n{data['ddl']}\n```")
    prompt_parts.append("#### Sample Rows:")
    prompt_parts.append(f"```markdown\n{data['samples']}\n```")
    prompt_parts.append("-" * 40)

prompt_parts.append("\nOutput only the clean SQL INSERT statements inside a single code block.")

final_prompt = "\n".join(prompt_parts)
print(final_prompt)

You are a database assistant. Generate realistic SQL INSERT statements for SQL Server.
Please generate 10 new, realistic records for each of the tables described below.
Ensure correct referential integrity between tables.

### Table: Report.vw_CBS
#### Schema:
```sql
CREATE TABLE Report.vw_CBS (
    CBSKEY INTEGER NOT NULL,
    CBSName NVARCHAR(1000) COLLATE "SQL_Latin1_General_CP1_CI_AS" NOT NULL,
    OrganizationUnitPK INTEGER NULL,
    OrganUnitName NVARCHAR(500) COLLATE "SQL_Latin1_General_CP1_CI_AS" NULL,
    Stakeholder_ParentKEY INTEGER NULL,
    CreatedBy INTEGER NULL,
    LastModifiedBy INTEGER NULL,
    CreationTimeKEY BIGINT NULL,
    LastModificationTimeKEY BIGINT NULL,
    Weight FLOAT NULL,
    ProjectID INTEGER NULL,
    Budget FLOAT NULL,
    IRRAmmount FLOAT NULL,
    USDAmmount FLOAT NULL,
    EURAmmount FLOAT NULL,
    AEDAmmount FLOAT NULL,
    WBSID INTEGER NULL
);
```
#### Sample Rows:
```markdown
|   CBSKEY | CBSName      |   OrganizationUnitPK | OrganUnitName   

In [27]:
print("\n--- STEP 1: Running SQL Database Inspection ---")
extracted_metadata = pipeline.extract_database_metadata(
    schema_names=["Report"], 
    output_dir='../data/clean/metadata',
    columns_desc_csv_path="../docs/test"
)


--- STEP 1: Running SQL Database Inspection ---

--- Processing schema 'Report' ---
Found 33 objects in schema 'Report'. Extracting metadata and value samples...
   - Skipping excluded column from database metadata: Report.vw_Cashflow.DateType
   - Skipping excluded column from database metadata: Report.vw_Cashflow.JFromDate
   - Skipping excluded column from database metadata: Report.vw_Cashflow.JToDate
 - Saved: ../data/clean/metadata/Report_vw_Cashflow.json
   - Skipping excluded column from database metadata: Report.vw_CBS.CreatedBy
   - Skipping excluded column from database metadata: Report.vw_CBS.LastModifiedBy
 - Saved: ../data/clean/metadata/Report_vw_CBS.json
   - Skipping excluded column from database metadata: Report.vw_Contracts.ContractedStartDate_Year
   - Skipping excluded column from database metadata: Report.vw_Contracts.ContractedStartDate_Month
   - Skipping excluded column from database metadata: Report.vw_Contracts.ContractedStartDate_Day
   - Skipping excluded c

   - Skipping excluded column from database metadata: Report.vw_Projects.اسکلت سازه
   - Skipping excluded column from database metadata: Report.vw_Projects.برچسب
   - Skipping excluded column from database metadata: Report.vw_Projects.بهره بردار
   - Skipping excluded column from database metadata: Report.vw_Projects.پیوست دارد
   - Skipping excluded column from database metadata: Report.vw_Projects.تست date
   - Skipping excluded column from database metadata: Report.vw_Projects.تست طراحی
   - Skipping excluded column from database metadata: Report.vw_Projects.تست عدد
   - Skipping excluded column from database metadata: Report.vw_Projects.روش اداره ی پروژه
   - Skipping excluded column from database metadata: Report.vw_Projects.سبد
   - Skipping excluded column from database metadata: Report.vw_Projects.سطح تصویت
   - Skipping excluded column from database metadata: Report.vw_Projects.شرح نیاز مشتری
   - Skipping excluded column from database metadata: Report.vw_Projects.فیلد1
   - 

In [28]:
print("\n--- STEP 1.5: Exporting Column Description Template CSV ---")
COLUMNS_DESC_CSV = '../docs/clean/column_desc.csv'
# Generate the template if it doesn't already exist so the user can populate it
if extracted_metadata:
    pipeline.export_column_desc_template(extracted_metadata, COLUMNS_DESC_CSV)


--- STEP 1.5: Exporting Column Description Template CSV ---
Successfully generated column description template: '../docs/clean/column_desc.csv'


In [34]:
print("--- STEP 0: Converting View Description CSV to JSON ---")
VIEWS_CSV = '../docs/clean/view_descriptions.csv'  # Path to your views CSV
VIEWS_JSON = '../data/clean/view_descriptions.json'  # Target JSON file output

# Check if views CSV exists, if so convert it
view_desc_dict = {}
if os.path.exists(VIEWS_CSV):
    view_desc_dict = pipeline.convert_view_desc_csv_to_json(VIEWS_CSV, VIEWS_JSON)
else:
    print(f"Skipping Step 0: '{VIEWS_CSV}' not found. Place your View/Description CSV here.")

--- STEP 0: Converting View Description CSV to JSON ---
Skipping Step 0: '../docs/clean/view_descriptions.csv' not found. Place your View/Description CSV here.


In [29]:
print("\n--- STEP 0.5: Converting Column Description CSV to JSONL ---")
COLUMNS_DESC_CSV = '../docs/clean/column_desc.csv'
COLUMNS_DESC_JSONL = '../data/clean/column_description.jsonl'

if os.path.exists(COLUMNS_DESC_CSV):
    pipeline.convert_column_desc_csv_to_jsonl(COLUMNS_DESC_CSV, COLUMNS_DESC_JSONL)
else:
    print(f"Skipping Step 0.5: '{COLUMNS_DESC_CSV}' not found.")


--- STEP 0.5: Converting Column Description CSV to JSONL ---
Successfully converted CSV and saved JSONL to: '../data/clean/column_description.jsonl'


In [35]:
print("\n--- STEP 2: Building ChromaDB Search Index ---")

# if os.path.exists(COLUMNS_DESC_CSV):
#     print(f"Populating from user's key file: '{COLUMNS_DESC_CSV}'")
#     pipeline.create_chroma_db_from_csv(COLUMNS_DESC_CSV)
# else:
#     print(f"'{COLUMNS_DESC_CSV}' not found.")
# FALLBACK: Automatically build the vector database using SQL schema, 
# now enriched with the view descriptions we mapped in Step 0!
# print("Fallback: Dynamically indexing vector database from live DB metadata with view contextual descriptions...")
pipeline.create_chroma_db_from_metadata(
    metadata_registry=extracted_metadata, 
    view_desc_dict=view_desc_dict,
    batch_size=5
)


--- STEP 2: Building ChromaDB Search Index ---
Initializing ChromaDB client at '../data/clean/schema_db'...
Ollama detected. Using local model: embeddinggemma
Writing 410 items in batches of 5...
Sample 0 of 410
Sample 5 of 410
Sample 10 of 410
Sample 15 of 410
Sample 20 of 410
Sample 25 of 410
Sample 30 of 410
Sample 35 of 410
Sample 40 of 410
Sample 45 of 410
Sample 50 of 410
Sample 55 of 410
Sample 60 of 410
Sample 65 of 410
Sample 70 of 410
Sample 75 of 410
Sample 80 of 410
Sample 85 of 410
Sample 90 of 410
Sample 95 of 410
Sample 100 of 410
Sample 105 of 410
Sample 110 of 410
Sample 115 of 410
Sample 120 of 410
Sample 125 of 410
Sample 130 of 410
Sample 135 of 410
Sample 140 of 410
Sample 145 of 410
Sample 150 of 410
Sample 155 of 410
Sample 160 of 410
Sample 165 of 410
Sample 170 of 410
Sample 175 of 410
Sample 180 of 410
Sample 185 of 410
Sample 190 of 410
Sample 195 of 410
Sample 200 of 410
Sample 205 of 410
Sample 210 of 410
Sample 215 of 410
Sample 220 of 410
Sample 225 of 4

In [9]:
lsh_columns_to_index = {
    "Report": {
        "vw_CBS": [
            "CBSName",
            "OrganUnitName"
        ],
        "vw_Contracts": [
            # "کد قرارداد",
            # "شماره قرارداد",
            "عنوان قرارداد",
            # "کد پروژه",
            "نام پروژه",
            "وضعیت قرارداد",
            "نام کارگاه - فرم گردش قراداد",
            # "شماره طرف قرارداد"
        ],
        "vw_DataFormItem": [
            "ProjectName",
            "DataFormItemTitle",
            "DisciplineName",
            "DFITypeName",
            "WorkflowModelName",
            "AssignedUsers"
        ],
        # "vw_FB_شناسنامه_احکام_و_گزارش_عملکرد_سازمان_تامین_اجتماعی_در_برنامه_هفتم_پیشرفت#شناسنامه_حکم": [
        #     "شناسه_حکم",
        #     "موضوع_"
        # ],
        # "vw_FB_شناسنامه_احکام_و_گزارش_عملکرد_سازمان_تامین_اجتماعی_در_برنامه_هفتم_پیشرفت#واحد_متولی_معاونت": [
        #     "واحد_همکار",
        #     "واحد_همکار_درون_سازمانی_",
        #     "دستگاه_متولی_برون_سازمانی_",
        #     "واحد_اصلی"
        # ],
        # "vw_FB_فرم_آپلود_فایل": [
        #     "نام_و_نام_خانوادگی"
        # ],
        # "vw_FB_فرم_برنامه_هفتم": [
        #     "واحد_همکار_درون_سازمانی_",
        #     "شناسه_حکم",
        #     "موضوع_",
        #     "دستگاه_متولی_برون_سازمانی_"
        # ],
        # "vw_FB_فرم_برنامه_هفتم#عنوان_برنامه_عملیاتی": [
        #     "عنوان_برنامه_عملیاتی"
        # ],
        "vw_Invoice": [
            "ContractTitle",
            "InvoiceNumber",
            "CBSName"
        ],
        "vw_Issues": [
            "OrganUnitName",
            "IssuesTitle",
            "Severity",
            "AssignedUserDetailsFullName",
            "CreatorUserDetailsFullName",
            "LastModifierUserDetailsFullName",
            "ProjectIssueStatusName",
            "ProjectName",
            "ProjectIssueCategoriesTitle",
            "StakeholderName"
        ],
        "vw_OrganizationUnit": [
            "OrganizationUnitTitle"
        ],
        "vw_OrganizationUnitType": [
            "OrganizationUnitTypeTitle"
        ],
        "vw_OrganUnitScurve": [
            "OrgTitle"
        ],
        "vw_Payments": [
            "OrganUnitName",
            # "DocumentNumber",
            "CurrencyName",
            # "InvoiceNumber",
            "ProjectName",
            "CBSName",
            "ContractTitle",
            "CostCenterName",
            "PaymentBehalfName",
            "PaymentTypeName"
        ],
        "vw_PlanHistory": [
            "ProjectName",
            "wbsName",
            "ProjectPlanName"
        ],
        "vw_ProjectGroup": [
            "ProjectGroupName"
        ],
        "vw_ProjectMonthlyReport": [
            "PrjName",
            "StatusTitle",
            # "ReportNumber"
        ],
        "vw_ProjectPlan": [
            "ProjectPlanName",
            "ProjectName"
        ],
        "vw_ProjectPlanProgress": [
            # "کد پروژه",
            "نام پروژه"
        ],
        "vw_Projects": [
            "PMFullName",
            "TaskMaster",
            "Contractor",
            "Supervisor",
            # "ContractNo",
            # "ProjectNO",
            "ContractSubject",
            "ProjectManager",
            "Consultant",
            "Client",
            "DefaultLocationTitle",
            # "کد پروژه",
            "نام پروژه",
            "نوع پروژه",
            "وضعیت پروژه",
            "واحد سازمانی پروژه",
            "کشور پروژه",
            "استان پروژه",
            "شهر پروژه",
            "اسکلت سازه",
            "برچسب",
            "بهره بردار",
            "روش اداره ی پروژه",
            "سبد",
            "سطح تصویت",
            "ماهیت",
            "ProjectTypeName",
            "CurrencyName",
            "ProjectWBSName"
        ],
        "vw_ResourceUsage": [
            "ResourceName",
            "ResourceTypeName",
            "ResourceGroupName",
            "WBSName",
            # "WBSCode",
            # "ActivityCode",
            # "WBSPath"
        ],
        "vw_Risks": [
            "OrganUnitName",
            "RiskTitle",
            "Severity",
            "AssignedUserDetailsFullName",
            "CreatorUserDetailsFullName",
            "LastModifierUserDetailsFullName",
            "ProjectName",
            "ProjectRiskCategoriesTitle",
            "RiskStatusName",
            "StakeholderName"
        ],
        "vw_WBS": [
            # "Code",
            "Name",
            # "ActivityCode",
            "ProjectName",
            # "ParentPath"
        ],
        "vw_WorkItems": [
            "نام پروژه",
            # "کد پروژه",
            # "کد قلم کاری",
            "عنوان قلم کاری",
            "نام دیسیپلین قلم کاری",
            "نام چرخه کاری قلم کاری",
            "نوع قلم کاری",
            "استان",
            "سایت",
            "شهر",
            "مسول"
        ]
    }
}

In [13]:
lsh_columns_to_index = {
    "Report": {
        # "vw_Cashflow": [
        #     "GFromDate",
        #     "GToDate",
        #     "PlanCashIn",
        #     "PlanCashOut",
        #     "TotalPlanCash",
        #     "PlanCashInCumulative",
        #     "PlanCashOutCumulative",
        #     "PlanCashCumulative",
        #     "ActualCashIn",
        #     "ActualCashOut",
        #     "TotalActualCash",
        #     "ActualCashInCommulative",
        #     "ActualCashOutCommulative",
        #     "TotalActualCashCommulative",
        #     "ProjectID",
        #     "TimePeriod"
        # ],
        "vw_CBS": [
        #     "CBSID",
            "CBSName",
        #     "OrganizationUnitID",
            "OrganUnitName",
        #     "Stakeholder_ParentID",
        #     "CreationTime",
        #     "LastModificationTime",
        #     "Weight",
        #     "ProjectID",
        #     "Budget",
        #     "IRRAmmount",
        #     "USDAmmount",
        #     "EURAmmount",
        #     "AEDAmmount",
        #     "WBSID"
        ],
        "vw_Contracts": [
        #     "Contractkey",
        #     "ContractNumber",
            "ContractTitle",
            "IsCashIn",
        #     "ProjectKEY",
            "ProjectName",
        #     "ContractStatusName",
        #     "ContractedStartDate_Year",
        #     "ContractedStartDate_Month",
        #     "ContractedStartDate_Day",
        #     "ContractedStartDate_PersianDate",
        #     "ContractedStartDate_MiladiDate",
        #     "ContractFinishDate_Year",
        #     "ContractFinishDate_Month",
        #     "ContractFinishDate_Day",
        #     "ContractFinishDate_PersianDate",
        #     "ContractFinishDate_MiladiDate",
        #     "ActualProgress",
        #     "PlanProgress",
        #     "ContractAmount",
        #     "ContractFinalAmount",
        #     "SumContractAmendment",
        #     "ContractDate_Year",
        #     "ContractDate_Month",
        #     "ContractDate_Day",
        #     "ContractDate_MiladiDate",
        #     "AmountPayment",
        #     "AmountInvoice",
        #     "CountContractAmendment",
        #     "CountInvoice",
        #     "MaxAmendmentDate_Year",
        #     "MaxAmendmentDate_Month",
        #     "MaxAmendmentDate_Day",
        #     "MaxAmendmentDate_MiladiDate",
        #     "MaxInvoice_Year",
        #     "MaxInvoice_Month",
        #     "MaxInvoice_Day",
        #     "MaxInvoice_MiladiDate",
        #     "CountPayment",
        #     "MaxPaiementDate_Year",
        #     "MaxPaiementDate_Month",
        #     "MaxPaiementDate_Day",
        #     "MaxPaiementDate_MiladiDate",
        #     "پیوست گردش قراداد مطالعه شد.",
        #     "تاریخ شروع کار پیمانکار - فرم گردش قرارداد",
        #     "تضمین انجام تعهدات",
        #     "شماره طرف قرارداد",
        #     "شماره فرم گردش قرارداد",
        #     "مدت دوره تضمین - فرم گردش قرارداد",
        #     "نام کارگاه - فرم گردش قراداد"
        ],
        # "vw_FB_شناسنامه_احکام_و_گزارش_عملکرد_سازمان_تامین_اجتماعی_در_برنامه_هفتم_پیشرفت#برنامه_عملیاتی_": [
        #     "ID",
        #     "ParentID",
        #     "ProjectID",
        #     "در_صورت_داشتن_برنامه_عملیاتی"
        # ],
        # "vw_FB_شناسنامه_احکام_و_گزارش_عملکرد_سازمان_تامین_اجتماعی_در_برنامه_هفتم_پیشرفت#شناسنامه_حکم": [
        #     "ID",
        #     "ParentID",
        #     "ProjectID",
        #     "متن_حکم",
        #     "شناسه_حکم",
        #     "موضوع_"
        # ],
        # "vw_FB_شناسنامه_احکام_و_گزارش_عملکرد_سازمان_تامین_اجتماعی_در_برنامه_هفتم_پیشرفت#هدف_کمی": [
        #     "ID",
        #     "ParentID",
        #     "ProjectID",
        #     "سال_پنجم",
        #     "در_صورت_داشتن_هدف_کمی__میزان_تحقق_در_سال_اول_",
        #     "سال_چهارم",
        #     "سال_دوم",
        #     "سال_سوم"
        # ],
        # "vw_FB_شناسنامه_احکام_و_گزارش_عملکرد_سازمان_تامین_اجتماعی_در_برنامه_هفتم_پیشرفت#واحد_متولی_معاونت": [
        #     "ID",
        #     "ParentID",
        #     "ProjectID",
        #     "واحد_همکار",
        #     "واحد_همکار_درون_سازمانی_",
        #     "دستگاه_متولی_برون_سازمانی_",
        #     "واحد_اصلی"
        # ],
        # "vw_FB_فرم_آپلود_فایل": [
        #     "ID",
        #     "ParentID",
        #     "ProjectID",
        #     "نام_و_نام_خانوادگی"
        # ],
        # "vw_FB_فرم_اقای_احمدی": [
        #     "ID",
        #     "ParentID",
        #     "ProjectID",
        #     "متن"
        # ],
        # "vw_FB_فرم_اقای_احمدی#child": [
        #     "ID",
        #     "ParentID",
        #     "ProjectID",
        #     "میزان_تحقق_"
        # ],
        # "vw_FB_فرم_برنامه_هفتم": [
        #     "ID",
        #     "ParentID",
        #     "ProjectID",
        #     "واحد_همکار_درون_سازمانی_",
        #     "توضیحات_",
        #     "متن_حکم",
        #     "موضوع_",
        #     "در_صورت_داشتن_برنامه_عملیاتی_",
        #     "اصلی_",
        #     "همکار_",
        #     "دستگاه_متولی_برون_سازمانی_"
        # ],
        # "vw_FB_فرم_برنامه_هفتم#عملکرد_حکم": [
        #     "ID",
        #     "ParentID",
        #     "ProjectID",
        #     "زمان_اقدام",
        #     "واحد_سنجش",
        #     "خروجی_مورد_انتظار_",
        #     "میزان_تحقق_خروجی",
        #     "اقدامات_آتی",
        #     "شرح_اقدامات_حوزه_همکار_",
        #     "موانع_اجرایی",
        #     "شرح_اقدامات_حوزه_متولی_"
        # ],
        # "vw_FB_فرم_برنامه_هفتم#عنوان_برنامه_عملیاتی": [
        #     "ID",
        #     "ParentID",
        #     "ProjectID",
        #     "شرح_فعالیت",
        #     "خروجی",
        #     "زمان_پایان",
        #     "عنوان_برنامه_عملیاتی",
        #     "درصد_پیشرفت",
        #     "زمان_شروع_"
        # ],
        # "vw_FB_فرم_برنامه_هفتم#هدف_کمی": [
        #     "ID",
        #     "ParentID",
        #     "ProjectID",
        #     "میزان_تحقق_"
        # ],
        "vw_Invoice": [
        #     "InvoiceKey",
        #     "ContractPK",
            "ContractTitle",
        #     "InvoiceNumber",
        #     "DateReceived",
        #     "DateIssued",
        #     "Amount",
        #     "PreviousAccepted",
        #     "FinalAcceptedAmount",
        #     "CBSID",
            "CBSName",
        #     "ValueAddedTax",
        #     "IsInvoiceBalance",
        #     "BalanceInvoiceNumber"
        ],
        "vw_Issues": [
        #     "IssueID",
        #     "OrganizationUnitID",
            "IssuesTitle",
            "Description",
            "ActionDescription",
        #     "Severity",
        #     "IdentificationDate",
        #     "DueDate",
            "Resolution",
        #     "AssignedUserDetailsUserName",
        #     "AssignedUserDetailsFirstName",
        #     "AssignedUserDetailsLastName",
        #     "IssueStatus",
        #     "CreatorUserDetailsUserName",
        #     "CreatorUserDetailsFirstName",
        #     "CreatorUserDetailsLastName",
        #     "LastModifierUserDetailsUserName",
        #     "LastModifierUserDetailsFirstName",
        #     "LastModifierUserDetailsLastName",
        #     "ProjectID",
            "ProjectName",
        #     "IssueCategory",
            "AssignedUserDetailsFullName",
            "CreatorUserDetailsFullName",
            "LastModifierUserDetailsFullName",
        #     "ReferCount",
            "ResponsibleStakeholder"
        ],
        "vw_OrganizationUnit": [
        #     "StakeholderID",
            "OrganizationUnitTitle",
        #     "ParentID",
        #     "ProgProgress",
        #     "ActualProgress"
        ],
        "vw_OrganizationUnitType": [
        #     "OrganizationUnitTypeID",
            "OrganizationUnitTypeTitle"
        ],
        "vw_OrganUnitScurve": [
        #     "FromDate",
        #     "Todate",
            "OrganizationUnitTitle",
        #     "ParentId",
        #     "PlanProgress",
        #     "ActualProgress",
        #     "TimePeriodtxt"
        ],
        "vw_Payments": [
        #     "PaymentID",
        #     "OrganizationUnitID",
            "OrganUnitName",
        #     "ParentID",
        #     "DocumentNumber",
        #     "PaymentDate",
        #     "Amount",
            "CurrencyName",
        #     "InvoiceID",
        #     "InvoiceNumber",
        #     "ProjectID",
            "ProjectName",
        #     "CBSID",
            "CBSName",
        #     "ContractID",
            "CostCenterName",
            "PaymentBehalfName",
            "PaymentTypeName"
        ],
        # "vw_Photos": [
        #     "PhotoID",
        #     "ProjectID",
        #     "PhotoValue"
        # ],
        "vw_PlanHistory": [
        #     "PlanHistoryID",
            "ProjectName",
        #     "WBSID",
            "wbsName",
        #     "ProjectPlanID",
            "ProjectPlanName",
        #     "CreationTime",
        #     "PlanStartDate",
        #     "PlanEndDate"
        ],
        "vw_ProjectGroup": [
        #     "ProjectGroupID",
            "ProjectGroupName"
        ],
        # "vw_ProjectGroupScurve": [
        #     "DateType",
        #     "FromDate",
        #     "ToDate",
        #     "PlanProgress",
        #     "ActualProgress",
        #     "PlanCommulative",
        #     "ActualCommulative",
        #     "ProjectGroupID",
        #     "TimePeriod"
        # ],
        "vw_ProjectMonthlyReport": [
        #     "ProjectMonthlyReportID",
        #     "ProjectID",
            "ProjectName",
        #     "StatusTitle",
        #     "CreationTime",
            "DoneWorkTitle",
            "DoneWorkWorkDescription",
            "FutureWorkTitle",
            "FutureWorkDescription",
        #     "IsDone",
        #     "ReportNumber"
        ],
        "vw_ProjectPlan": [
        #     "ProjectPlanID",
            "ProjectPlanName",
        #     "ProjectID",
            "ProjectName",
        #     "isDefault"
        ],
        "vw_ProjectPlanProgress": [
        #     "ProjectID",
            "ProjectName",
        #     "Date",
        #     "ActualProgress",
        #     "PlanProgress"
        ],
        "vw_Projects": [
        #     "ProjectId",
        #     "OrganizationUnitID",
            "PMFullName",
        #     "PMUserName",
        #     "Client",
        #     "Contractor",
        #     "Supervisor",
        #     "ContractNo",
        #     "ProjectNO",
            "ContractSubject",
        #     "CreatedBy",
        #     "LastModifiedBy",
        #     "CreationTime",
        #     "CreationTime_J",
        #     "LastModificationTime",
        #     "LastModificationTime_J",
        #     "ProjectAdmin",
        #     "DefaultLocation_CityID",
        #     "Location_Latitude",
        #     "Location_Longtitude",
            "Consultant",
        #     "DefaultLocationID",
        #     "LocationTitle",
            "ProjectName",
        #     "ProjectTypeKEY",
        #     "ProjectStatusName",
            "OrganizationUnitName",
        #     "ActualProgress",
        #     "PlanProgress",
        #     "ProgressOfBeginningOfYear_Plan",
        #     "ProgressOfEndOfYear_Plan",
        #     "ProgressOfBeginningOfYear_Actual",
        #     "ProgressOfEndOfYear_Actual",
        #     "RandTotal",
        #     "RandYear",
            "ProjectCountry",
            "ProjectProvince",
            "ProjectCity",
        #     "ParentID",
        #     "WeightOfParentProject",
        #     "ProjectHasControlPoints",
        #     "اسکلت سازه",
        #     "برچسب",
        #     "بهره بردار",
        #     "پیوست دارد",
        #     "تست date",
        #     "تست طراحی",
        #     "تست عدد",
        #     "روش اداره ی پروژه",
        #     "سبد",
        #     "سطح تصویت",
        #     "شرح نیاز مشتری",
        #     "فیلد1",
        #     "ماهیت",
        #     "مبلغ اولیه تأمین شده",
        #     "مدت اجرای پروژه (تخمین) ",
        #     "مدیر پروژه",
        #     "نیروی انسانی مورد نیاز",
        #     "هدف",
        #     "کابر 1",
        #     "کاربر 2",
        #     "روش اداره ی پروژه (از نحوه)",
        #     "چارچوب",
            "CurrencyName",
        #     "ProjectWBSKEY",
            "ProjectWBSName"
        ],
        # "vw_ProjectScurve": [
        #     "FromDate",
        #     "ToDate",
        #     "PlanProgress",
        #     "ActualProgress",
        #     "PlanCommulative",
        #     "ActualCommulative",
        #     "ProjectID",
        #     "TimePeriod"
        # ],
        "vw_ResourceUsage": [
        #     "VolUnit",
        #     "ProjectID",
        #     "ResourceID",
        #     "UsageAmount",
            "Comments",
        #     "Cost",
        #     "WBSID",
        #     "Abbreviation",
            "ResourceName",
        #     "ResourceTypeName",
        #     "ResourceGroupName",
            "WBSName",
        #     "WBSCode",
        #     "ActivityCode",
        #     "WBSPath",
        #     "ActualStartDate"
        ],
        "vw_Risks": [
        #     "RiskID",
        #     "OrganizationUnitID",
            "OrganUnitName",
        #     "ParentID",
            "RiskTitle",
            "Description",
        #     "Severity",
        #     "IdentificationDate",
        #     "DueDate",
            "Resolution",
        #     "AssignedTo",
        #     "Type",
        #     "Probability",
        #     "DetectionCapability",
        #     "Sign",
        #     "SupportResponse",
        #     "AssignedUserDetailsUserName",
            "AssignedUserDetailsFirstName",
            "AssignedUserDetailsLastName",
        #     "CreatorUserDetailsUserName",
            "CreatorUserDetailsFirstName",
            "CreatorUserDetailsLastName",
        #     "LastModifierUserDetailsUserName",
            "LastModifierUserDetailsFirstName",
            "LastModifierUserDetailsLastName",
        #     "ProjectID",
            "ProjectName",
        #     "RiskCategory",
        #     "RiskStatus",
        #     "ReferCount",
            "ActionDescription",
        #     "OrganizationUnit"
        ],
        "vw_WBS": [
        #     "ParentID",
        #     "WbsId",
        #     "Duration",
        #     "DurationUnit",
        #     "IsActivity",
        #     "Code",
            "Name",
        #     "ActivityCode",
        #     "PlanEndDate",
        #     "PlanStartDate",
        #     "ActualEndDate",
        #     "ActualStartDate",
        #     "IsMilestone",
        #     "ProjectID",
            "ProjectName",
        #     "Weight",
        #     "Critical",
        #     "PlanProgress",
        #     "ActualProgress",
        #     "FieldProgress",
        #     "lvl",
        #     "ParentPath"
        ],
        "vw_WorkItems": [
        #     "WorkItemID",
        #     "WBSID",
        #     "WeightOfWhole",
        #     "ProjectID",
            "ProjectName",
            "WorkItemTitle",
        #     "Duration",
        #     "DurationUnit",
        #     "ActualStartDateKEY",
        #     "ActualStartDate_Year",
        #     "ActualStartDate_Month",
        #     "ActualStartDate_Day",
        #     "ActualStartDate_PersianDate",
        #     "ActualStartDate",
        #     "ActualEndDateKEY",
        #     "ActualEndDate_Year",
        #     "ActualEndDate_Month",
        #     "ActualEndDate_Day",
        #     "ActualEndDate_PersianDate",
        #     "ActualEndDate",
        #     "PlanStartDateKEY",
        #     "PlanStartDate_Year",
        #     "PlanStartDate_Month",
        #     "PlanStartDate_Day",
        #     "PlanStartDate_PersianDate",
        #     "PlanStartDate",
        #     "PlanEndDateKEY",
        #     "PlanEndDate_Year",
        #     "PlanEndDate_Month",
        #     "PlanEndDate_Day",
        #     "PlanEndDate_PersianDate",
        #     "PlanEndDate",
        #     "ActualProgress",
        #     "PlanProgress",
            "DisciplineName",
        #     "WorkItemType",
        #     "WorkflowModel",
        #     "VolumnOfWork",
        #     "VolUnitAbbr",
        #     "VolUnit",
        #     "SubProjectID",
            "ResponsibleUser"
        ]
    }
}

In [14]:
pipeline.build_value_lsh_index(
    lsh_targets=lsh_columns_to_index,
    threshold=0.4,
    shingle_size=3,
    sample_limit=10000
)

Fetching and indexing values for: Report.vw_CBS.CBSName...
   ! Values skip for Report.vw_CBS.CBSName: The given key already exists
Fetching and indexing values for: Report.vw_CBS.OrganUnitName...
Fetching and indexing values for: Report.vw_Contracts.ContractTitle...
Fetching and indexing values for: Report.vw_Contracts.IsCashIn...
Fetching and indexing values for: Report.vw_Contracts.ProjectName...
Fetching and indexing values for: Report.vw_Invoice.ContractTitle...
Fetching and indexing values for: Report.vw_Invoice.CBSName...
Fetching and indexing values for: Report.vw_Issues.IssuesTitle...
Fetching and indexing values for: Report.vw_Issues.Description...
Fetching and indexing values for: Report.vw_Issues.ActionDescription...
Fetching and indexing values for: Report.vw_Issues.Resolution...
Fetching and indexing values for: Report.vw_Issues.ProjectName...
Fetching and indexing values for: Report.vw_Issues.AssignedUserDetailsFullName...
Fetching and indexing values for: Report.vw_Issu

In [15]:

# 4. Perform search / query (useful when processing natural language questions)
# Suppose the user types "Acm Corp" but the database actually has "Acme Corporation"
results = pipeline.query_lsh("ناصر اسدی")

# Output matches sorted by exact Jaccard similarity descending:
for match in results[:5]:
    print(f"Match found in '{match['schema']}.{match['table']}.{match['column']}':")
    print(f"  -> Actual Value: {match['value']}")
    print(f"  -> Match Confidence (Jaccard): {match['similarity']:.4f}\n")

Match found in 'Report.vw_WorkItems.ResponsibleUser':
  -> Actual Value: ناصر اسدی
  -> Match Confidence (Jaccard): 1.0000

Match found in 'Report.vw_Issues.AssignedUserDetailsFullName':
  -> Actual Value: ناصر اسدی
  -> Match Confidence (Jaccard): 1.0000

Match found in 'Report.vw_Issues.LastModifierUserDetailsFullName':
  -> Actual Value: ناصر اسدی
  -> Match Confidence (Jaccard): 1.0000

Match found in 'Report.vw_Projects.PMFullName':
  -> Actual Value: ناصر اسدی
  -> Match Confidence (Jaccard): 1.0000

Match found in 'Report.vw_Issues.CreatorUserDetailsFullName':
  -> Actual Value: ناصر اسدی
  -> Match Confidence (Jaccard): 1.0000



In [31]:
print("\n--- STEP 3: Verification Query ---")
client = chromadb.PersistentClient(path='../data/clean/schema_db')
try:
    col = client.get_collection(name='schema_collection')
    query_word = "project name"
    print(f"Testing search for: '{query_word}'")
    results = col.query(query_texts=[query_word], n_results=3)
    
    for i, text_doc in enumerate(results['documents'][0]):
        meta = results['metadatas'][0][i]
        print(f"  Result {i+1}:")
        print(f"    Document: {text_doc}")
        print(f"    Metadata: {meta}")
except Exception as e:
    print(f"Could not fetch tests: {e}")


--- STEP 3: Verification Query ---
Testing search for: 'project name'
  Result 1:
    Document: Column 'ProjectName' in table 'Report.vw_ProjectMonthlyReport'. Data Type: NVARCHAR(500) COLLATE "Latin1_General_CI_AS" (non-nullable). Sample/Categorical Values: ['IVS', 'پروژه ساختمانی قیطریه'].
    Metadata: {'source': 'live_db_metadata', 'view_name': 'Report.vw_ProjectMonthlyReport', 'view_column': 'ProjectName'}
  Result 2:
    Document: Column 'ProjectName' in table 'Report.vw_ProjectPlan'. Data Type: NVARCHAR COLLATE "Latin1_General_CI_AS" (nullable). Sample/Categorical Values: ['AQR Sample Project', 'Construction P'].
    Metadata: {'view_name': 'Report.vw_ProjectPlan', 'source': 'live_db_metadata', 'view_column': 'ProjectName'}
  Result 3:
    Document: Column 'ProjectName' in table 'Report.vw_ProjectPlanProgress'. Data Type: NVARCHAR COLLATE "Latin1_General_CI_AS" (nullable). Sample/Categorical Values: ['AQR Sample Project', 'Construction P'].
    Metadata: {'view_column': 'Projec

In [13]:
import pickle

with open('../data/lsh_index.pkl', 'rb') as f:
    data = pickle.load(f)

print("Type of loaded object:", type(data))
if isinstance(data, dict):
    print("Keys in dictionary:", list(data.keys())[:5])
    first_key = list(data.keys())[0] if data else None
    if first_key:
        print(f"Type of value for key '{first_key}':", type(data[first_key]))

Type of loaded object: <class 'dict'>
Keys in dictionary: ['lsh', 'num_perm', 'shingle_size', 'threshold']
Type of value for key 'lsh': <class 'datasketch.lsh.MinHashLSH'>


In [32]:
client.delete_collection(name='schema_collection')